In [ ]:
%pip install psycopg2-binary --quiet

In [ ]:
import boto3, json

dbutils.widgets.text("service_credential", "rex_prd_service_credential")
SERVICE_CREDENTIAL_NAME = dbutils.widgets.get("service_credential").strip() or "rex_prd_service_credential"
REGION = "eu-west-1"
SECRET_ID = "rds/rex-management/prod"

provider = dbutils.credentials.getServiceCredentialsProvider(SERVICE_CREDENTIAL_NAME)

session = boto3.Session(
    botocore_session=provider,
    region_name=REGION
)

sm = session.client("secretsmanager")
response = sm.get_secret_value(SecretId=SECRET_ID)

# Parse the secret - adjust keys based on actual secret structure
secret = json.loads(response["SecretString"]) if isinstance(response.get("SecretString"), str) else {"password": response["SecretString"]}
print("Secret keys available:", list(secret.keys()))

# The secret is the REX app's secret bundle (Cognitive/SendGrid/OpenAI keys live
# alongside), NOT an RDS credential: the Postgres details arrive as one
# Npgsql-style connection string, which is why there is no Password key on the
# secret itself. Fields are kept as written — Host, Port, Database, Username,
# Password — so the lookups in connect_pg() read the way the string does.
# Npgsql's client-side pool settings (Pooling, MinPoolSize, MaxPoolSize) are also
# present and deliberately ignored; psycopg2 has no equivalent.
creds = {}
for _part in secret["CONNECTION_STRING"].split(";"):
    if "=" in _part:
        _k, _v = _part.split("=", 1)
        creds[_k.strip()] = _v.strip().strip('"').strip("'")

# Field NAMES only — never the values, Password is in here. Note this print is
# visible in the Databricks UI run page but is NOT returned by
# `databricks jobs get-run-output`, which surfaces only error/error_trace.
print("Connection string fields:", sorted(creds))


In [ ]:
import psycopg2
import pandas as pd
from datetime import datetime, timezone

# ============================================================
# CONFIGURATION
# ============================================================
# Job parameters, with the dev values as defaults so the notebook still runs
# interactively. The job passes these so one notebook serves every target.
dbutils.widgets.text("target_catalog", "turbine_lens_dev")
dbutils.widgets.text("target_schema", "rex_prod")
TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip() or "turbine_lens_dev"
TARGET_SCHEMA = dbutils.widgets.get("target_schema").strip() or "rex_prod"

# Primary keys per table (based on REX EF Core model snapshot)
TABLE_PKS = {
    "Action": ["Id"],
    "ActionAssignee": ["Id"],
    "ActionShared": ["Id"],
    "ActionStatus": ["Id"],
    "Assessment": ["Id"],
    "CapexOpexRevenuesImpact": ["Id"],
    "CategoryFactor": ["Id"],
    "ChatMessages": ["Id"],
    "Country": ["Id"],
    "DisciplineWorkPackage": ["Id"],
    "DisciplineWorkPackageGlobalBusinessUnit": ["DisciplineWorkPackageId", "GlobalBusinessUnitId"],
    "GlobalBusinessUnit": ["Id"],
    "History": ["Id"],
    "ImpactDelay": ["Id"],
    "ImpactHse": ["Id"],
    "ImpactReputation": ["Id"],
    "ImportanceLevel": ["Id"],
    "KnowledgeArea": ["Id"],
    "Language": ["Id"],
    "LockedRex": ["RexId", "UserId"],
    "OrganisationUnit": ["Id"],
    "Permission": ["Id"],
    "Plant": ["Id"],
    "ProjectType": ["Id"],
    "ProjectTypeGlobalBusinessUnit": ["ProjectTypeId", "GlobalBusinessUnitId"],
    "Rankings": ["Id"],
    "Rex": ["Id"],
    "RexAssignee": ["Id"],
    "RexFile": ["Id"],
    "RexProjectPhase": ["Id"],
    "RexProjectPhaseGlobalBusinessUnit": ["RexProjectPhaseId", "GlobalBusinessUnitId"],
    "RexShared": ["Id"],
    "RexStatus": ["Id"],
    "RexTranslation": ["Id"],
    "Role": ["Id"],
    "RolePermission": ["RoleId", "PermissionId"],
    "Supplier": ["Id"],
    "SupplierGlobalBusinessUnit": ["SupplierId", "GlobalBusinessUnitId"],
    "TranslateResources": ["Id"],
    "User": ["Id"],
    "UserGlobalBusinessUnit": ["UserId", "GlobalBusinessUnitId"],
    "UserOrganisationUnit": ["UserId", "OrganisationUnitId"],
    "UserRex": ["UserId", "RexId"],
}

# ============================================================
# CONNECTION
# ============================================================
def connect_pg():
    return psycopg2.connect(
        # NOT creds["Host"], deliberately. The connection string names the raw RDS
        # endpoint (rex-management-tool-prod-db-instance-1...rds.amazonaws.com,
        # 127.0.0.1), which serverless job compute gets "Connection refused"
        # from — measured 2026-09-14. This internal alias (172.18.86.146) is the
        # route that is actually open, and it reached authentication on the same
        # day. Only the credentials come from the string.
        host="example.invalid",
        port=creds.get("Port", 5432),
        dbname=creds.get("Database", "rexmanagementtool"),
        user=creds.get("Username", ""),
        password=creds.get("Password", "")
    )

# ============================================================
# SYNC: Full Refresh + MERGE (one strategy for all tables)
# ============================================================
def sync_table(conn, table_name, pk_cols):
    """Read entire source table, MERGE into Delta target.
    Handles inserts, updates, and deletes in one pass.
    Returns a dict with operation metrics."""
    target = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{table_name}"

    # Full read from PostgreSQL
    pdf = pd.read_sql(f'SELECT * FROM public."{table_name}"', conn)

    if pdf.empty:
        print(f"  SKIP  {table_name} (0 rows in source)")
        return {"table": table_name, "inserted": 0, "updated": 0, "deleted": 0, "status": "skipped"}

    # First run: create table if it doesn't exist yet
    if not spark.catalog.tableExists(target):
        sdf = spark.createDataFrame(pdf)
        sdf.write.format("delta").mode("overwrite").saveAsTable(target)
        print(f"  CREATE {target} ({len(pdf)} rows) [initial load]")
        return {"table": table_name, "inserted": len(pdf), "updated": 0, "deleted": 0, "status": "created"}

    # Subsequent runs: MERGE (upsert + delete)
    sdf = spark.createDataFrame(pdf)
    sdf.createOrReplaceTempView(f"_src_{table_name}")

    merge_on = " AND ".join([f"target.`{pk}` = source.`{pk}`" for pk in pk_cols])
    non_pk_cols = [c for c in pdf.columns if c not in pk_cols]

    # Build MERGE statement.
    #
    # `UPDATE SET *` / `INSERT *` rather than an explicit column list, and that is what
    # makes SCHEMA EVOLUTION actually work. Evolution only adds a column when the clause
    # is the STAR form: naming `target.OstID` explicitly makes Spark resolve it against
    # the target BEFORE evolution can create it, so the statement dies at analysis with
    #   [DELTA_MERGE_UNRESOLVED_EXPRESSION] Cannot resolve target.OstID in UPDATE clause
    # even with `WITH SCHEMA EVOLUTION` present — measured 2026-09-15, where adding the
    # clause alone changed the error text from `given columns target.…` to
    # `given columns source.…` and otherwise failed identically. Two changes, one fix.
    #
    # The star form is safe here because source and target carry the same column names by
    # construction: the source view is a straight `SELECT *` of the Postgres table, so the
    # explicit lists this replaces were only ever spelling out "all columns".
    if non_pk_cols:
        merge_sql = f"""
            MERGE WITH SCHEMA EVOLUTION INTO {target} AS target
            USING _src_{table_name} AS source
            ON {merge_on}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
            WHEN NOT MATCHED BY SOURCE THEN DELETE
        """
    else:
        # Junction tables with only PK columns: nothing to update, so no UPDATE clause
        # (an `UPDATE SET *` here would just set the keys to themselves).
        merge_sql = f"""
            MERGE WITH SCHEMA EVOLUTION INTO {target} AS target
            USING _src_{table_name} AS source
            ON {merge_on}
            WHEN NOT MATCHED THEN INSERT *
            WHEN NOT MATCHED BY SOURCE THEN DELETE
        """

    # Execute MERGE and capture metrics
    result = spark.sql(merge_sql)
    metrics = result.collect()[0]
    
    inserted = metrics["num_inserted_rows"]
    updated = metrics["num_updated_rows"]
    deleted = metrics["num_deleted_rows"]

    # Print with details
    changes = []
    if inserted > 0:
        changes.append(f"+{inserted} inserted")
    if updated > 0:
        changes.append(f"~{updated} updated")
    if deleted > 0:
        changes.append(f"-{deleted} deleted")
    
    if changes:
        print(f"  MERGE {table_name:<30} {' | '.join(changes)}")
    else:
        print(f"  MERGE {table_name:<30} (no changes)")

    return {"table": table_name, "inserted": inserted, "updated": updated, "deleted": deleted, "status": "synced"}

# ============================================================
# RUN
# ============================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")

# Make the schema readable by the Turbine Lens group, in the same breath as creating it.
# The app's own service principal is a member, and it is the app that queries these
# tables through the REMA/REX Genie spaces at answer time — so without this the space
# answers nothing and does not error.
#
# It belongs HERE rather than in the bundle because the bundle does not own this schema:
# the notebook creates it, so the ingest SP owns it, and only the owner can grant. That
# is also the trap this line closes. Measured 2026-09-15: both schemas had been created
# INTERACTIVELY by a human, so `CREATE SCHEMA IF NOT EXISTS` was a no-op and the job SP
# could not write into a schema it did not own — 27 of 27 and 43 of 43 tables failed on
# `USE SCHEMA`. Ownership follows whoever creates first, so let it always be this job.
#
# GRANT is additive, unlike the bundle's declarative `grants:` which revokes anything
# undeclared, so re-running this cannot take a privilege away from anyone.
# Tolerated on purpose, and the ONLY failure in this notebook that is. Only a schema's
# owner may GRANT, and ownership follows whoever created it first: in prod this job
# creates the schema and owns it, so the grant lands; in noprod both schemas were created
# BY HAND long ago, so the job SP holds no MANAGE and this raises
# `PERMISSION_DENIED: User does not have MANAGE on Schema ...` — measured 2026-09-15,
# where it failed the whole nightly sync until this guard was added.
#
# Why tolerating THIS differs from tolerating a table failure: a missing grant cannot
# corrupt or stale the data, and it surfaces immediately and loudly the first time the app
# queries the space. A swallowed table failure does the opposite — it leaves a stale table
# looking current, which is how four REMA tables went two months unnoticed.
#
# The real fix is to converge ownership so both environments behave alike; until then this
# keeps a privilege nicety from taking down the data movement it is attached to.
try:
    spark.sql(
        f"GRANT USE SCHEMA, SELECT ON SCHEMA {TARGET_CATALOG}.{TARGET_SCHEMA} "
        f"TO `portfolio-org0004-default-turbine_lens`"
    )
    print(f"  GRANT USE SCHEMA, SELECT on {TARGET_CATALOG}.{TARGET_SCHEMA} -> portfolio-org0004-default-turbine_lens")
except Exception as _grant_err:
    print(f"  WARN  could not grant on {TARGET_CATALOG}.{TARGET_SCHEMA}: {_grant_err}")
    print("  WARN  this job does not own that schema; the app's read access must be granted "
          "by its owner. The sync itself is unaffected.")

run_start = datetime.now(timezone.utc)
print(f"{'='*70}")
print(f"FULL REFRESH + MERGE — {run_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"{'='*70}\n")

# Let MERGE absorb columns the source has gained.
# SCHEMA EVOLUTION is requested per-statement, in the MERGE below, via
# `MERGE WITH SCHEMA EVOLUTION INTO`. The obvious alternative does NOT work here:
# `spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")` is rejected
# outright on serverless compute —
#   [CONFIG_NOT_AVAILABLE.SERVERLESS_DELTA_SCHEMA_AUTO_MERGE_ENABLED] ... not supported in
#   serverless environments. Use the write option instead: .option("mergeSchema", "true")
# measured 2026-09-15 — and it fails at the CONF CALL, before any table is touched, so it
# takes down tasks that had nothing to do with schema drift. The suggested write option is
# a DataFrame-write option and does not apply to a SQL MERGE, which is what this notebook
# issues. The SQL clause is the serverless-safe equivalent.
# The MERGE below builds
# `UPDATE SET target.<col> = source.<col>` from the SOURCE columns, so a column added
# upstream makes the statement unresolvable against an existing target:
#   [DELTA_MERGE_UNRESOLVED_EXPRESSION] Cannot resolve target.OstID in UPDATE clause
# Measured 2026-09-15 on noprod `rex_prod.Rex` (36 columns) after `OstID` appeared in the
# source (37). The same day PROD synced the same table fine — not because prod was
# healthier, but because its table had just been created from scratch and therefore
# matched the source. Prod carried the identical latent bug, due on the next added column.
#
# What this does NOT do, deliberately: a column REMOVED upstream leaves a now-stale column
# on the target rather than dropping it, and an incompatible TYPE change still fails. Both
# are the right side of the trade — the first is visible and harmless, the second is a real
# incompatibility that should stop the run rather than be papered over.
conn = connect_pg()
all_metrics = []
failures = []

for table_name, pk_cols in TABLE_PKS.items():
    try:
        metrics = sync_table(conn, table_name, pk_cols)
        all_metrics.append(metrics)
    except Exception as e:
        print(f"  FAIL  {table_name}: {e}")
        failures.append((table_name, str(e)))

conn.close()

# ============================================================
# SUMMARY REPORT
# ============================================================
run_end = datetime.now(timezone.utc)
total_inserted = sum(m["inserted"] for m in all_metrics)
total_updated = sum(m["updated"] for m in all_metrics)
total_deleted = sum(m["deleted"] for m in all_metrics)
tables_with_changes = [m for m in all_metrics if m["inserted"] + m["updated"] + m["deleted"] > 0]

print(f"\n{'='*70}")
print(f"SYNC SUMMARY")
print(f"{'='*70}")
print(f"  Duration:     {(run_end - run_start).total_seconds():.1f}s")
print(f"  Tables:       {len(TABLE_PKS)} total, {len(tables_with_changes)} with changes")
print(f"  Inserted:     {total_inserted} rows")
print(f"  Updated:      {total_updated} rows")
print(f"  Deleted:      {total_deleted} rows")
if failures:
    print(f"  Failures:     {len(failures)}")
    for name, err in failures:
        print(f"    - {name}: {err}")
print(f"{'='*70}")

# Display as a summary table
if tables_with_changes:
    print(f"\n\nTables with changes:")
    changes_df = pd.DataFrame(tables_with_changes)
    changes_df = changes_df[changes_df['status'] != 'skipped']
    changes_df = changes_df.sort_values(by=['inserted', 'updated', 'deleted'], ascending=False)
    display(changes_df)

# ============================================================
# FAIL LOUDLY
# ============================================================
# The per-table loop above catches EVERY exception into `failures`, so without
# this the cell ends green no matter how much failed. Measured in prod
# 2026-09-15: all 27 tables failed, the task was reported as a successful sync
# of zero tables, and the only signal was the NEXT cell crashing on a table
# that had never been created. A scheduled run has no such witness, and this
# job carries no failure mail.
#
# Raises on ANY failure, not only a total one: every run is a full read plus
# MERGE so a retry converges, and the task already retries once — a table that
# fails twice is a real problem. A partially-synced schema feeding a Genie
# space is worse than a red job, because the space answers confidently from
# whatever did land.
#
# The reason goes in the exception MESSAGE on purpose: `databricks jobs
# get-run-output` returns error/error_trace but NOT the prints above, and the
# run export carries no cell output either — so a printed reason is invisible
# to everything except a human on the run page.
if failures:
    _detail = "; ".join(f"{_n}: {_e[:300]}" for _n, _e in failures[:5])
    _more = f" (+{len(failures) - 5} more)" if len(failures) > 5 else ""
    raise RuntimeError(
        f"{len(failures)} of {len(TABLE_PKS)} tables failed to sync into "
        f"{TARGET_CATALOG}.{TARGET_SCHEMA}; {len(all_metrics)} succeeded. "
        f"{_detail}{_more}"
    )


In [ ]:
# ============================================================
# VIEW DEFINITIONS — now created by the job
# ============================================================
# What the `rex_qualitative_context` Genie space reads. Previously inside a
# `/* ... */` block and therefore inert; verified byte-identical to the live noprod
# views before being enabled. See the equivalent cell in the REMA notebook.
_STATEMENTS = [
rf"""
CREATE OR REPLACE VIEW {TARGET_CATALOG}.{TARGET_SCHEMA}.v_rex_details AS
SELECT
    r.Id AS rex_id,
    r.GUID AS rex_guid,
    rt.Title,
    rt.WhatWentWrongOrWell,
    rt.CorrectiveActionOrGainsAchieved,
    rt.LessonLearned,
    r.RelevantForCountry,
    r.RelevantForGbu,
    r.RelevantForEntity,
    r.RelevantForPortfolio,
    r.EventDate,
    rt.LatestTranslationDate,
    rs.Code AS status_code,
    gbu.Code AS global_business_unit_code,
    a.Code AS assessment_code,
    dwp.Code AS discipline_work_package_code,
    pp.Code AS project_phase_code,
    ka.Code AS knowledge_area_code,
    cf.Code AS category_factor_code,
    ic.Code AS impact_capex_code,
    io.Code AS impact_opex_code,
    idl.Code AS impact_delay_code,
    irv.Code AS impact_revenues_code,
    ih.Code AS impact_hse_code,
    ir.Code AS impact_reputation_code,
    il.Code AS importance_level_code,
    s.Code AS supplier_code,
    p.Code AS plant_code,
    p.Name AS plant_name,
    p.CapacityMW AS plant_capacity_mw,
    c.Code AS country_code,
    c.Name AS country_name,
    pt.Code AS plant_type_code,
    pt.Name AS plant_type_name,
    creator.GUID AS creator_guid,
    creator.FirstName AS creator_first_name,
    creator.LastName AS creator_last_name,
    validator.GUID AS validated_by_guid,
    validator.FirstName AS validated_by_first_name,
    validator.LastName AS validated_by_last_name
FROM {TARGET_CATALOG}.{TARGET_SCHEMA}.Rex r
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.RexTranslation rt ON rt.RexGuid = r.GUID
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.RexStatus rs ON r.StatusID = rs.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.GlobalBusinessUnit gbu ON r.GlobalBusinessUnitID = gbu.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.Assessment a ON r.AssessmentID = a.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.DisciplineWorkPackage dwp ON r.DisciplineWorkPackageID = dwp.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.RexProjectPhase pp ON r.ProjectPhaseID = pp.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.KnowledgeArea ka ON r.KnowledgeAreaID = ka.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.CategoryFactor cf ON r.CategoryFactorID = cf.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.CapexOpexRevenuesImpact ic ON r.ImpactCapexID = ic.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.CapexOpexRevenuesImpact io ON r.ImpactOpexID = io.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.ImpactDelay idl ON r.ImpactDelayID = idl.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.CapexOpexRevenuesImpact irv ON r.ImpactRevenuesID = irv.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.ImpactHse ih ON r.ImpactHseID = ih.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.ImpactReputation ir ON r.ImpactReputationID = ir.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.ImportanceLevel il ON r.ImportanceLevelID = il.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.Supplier s ON r.SupplierID = s.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.Plant p ON r.PlantID = p.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.Country c ON r.CountryID = c.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.ProjectType pt ON r.PlantTypeID = pt.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.User creator ON r.CreatorID = creator.Id
LEFT JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.User validator ON r.ValidatedByID = validator.Id;

-- v_rex_translation_latest: equivalent intent of GetRexTranslationByGuid without input params.
-- Language priority in this example:
--   1) requested language id = 2 (French)
--   2) english language id = 1 (English)
--   3) default language flag
""",
rf"""
CREATE OR REPLACE VIEW {TARGET_CATALOG}.{TARGET_SCHEMA}.v_rex_translation_latest AS
WITH ranked AS (
    SELECT
        r.Id AS rex_id,
        r.GUID AS rex_guid,
        rt.LanguageId,
        rt.Title,
        rt.WhatWentWrongOrWell,
        rt.CorrectiveActionOrGainsAchieved,
        rt.LessonLearned,
        rt.LatestTranslationDate,
        ROW_NUMBER() OVER (
            PARTITION BY r.GUID
            ORDER BY
                CASE
                    WHEN rt.LanguageId = 2 THEN 0
                    WHEN rt.LanguageId = 1 THEN 1
                    WHEN rt.IsDefaultLanguageId = TRUE THEN 2
                    ELSE 3
                END,
                rt.LatestTranslationDate DESC
        ) AS rn
    FROM {TARGET_CATALOG}.{TARGET_SCHEMA}.Rex r
    JOIN {TARGET_CATALOG}.{TARGET_SCHEMA}.RexTranslation rt ON rt.RexGuid = r.GUID
)
SELECT
    rex_id,
    rex_guid,
    LanguageId,
    Title,
    WhatWentWrongOrWell,
    CorrectiveActionOrGainsAchieved,
    LessonLearned,
    LatestTranslationDate
FROM ranked
WHERE rn = 1
""",
]

for _i, _sql in enumerate(_STATEMENTS, 1):
    spark.sql(_sql)
    print(f"  [{_i}/{len(_STATEMENTS)}] applied")
